# The league of N investors: results of the saved networks

This notebook shows what the finite-N game (`src/mfg/pinn/nplayer.py`) found: N investors who
see each other's wealth, each penalised for ending below the **group's** mean, solved by one
symmetric network per N trained in rounds (fictitious play) and measured by Monte Carlo.

- **What it does:** loads the networks and the analysis saved by `scripts/run_nplayer.py` and
  `scripts/analyze_nplayer.py` in `results/nplayer/<run>/`, and shows the figures with a text
  before each one ("what is plotted") and after it ("what to look at").
- **What it does not do:** no training here (hours). To produce the results:
  `python scripts/run_nplayer.py` and then `python scripts/analyze_nplayer.py`, from the repo
  folder with `PYTHONPATH=src`.

The method is explained in `exploratory/guia_liga_n_inversores.html` (Spanish) and in the module
docstrings; the reasons for every setting are in `configs/nplayer.yaml`.

## 1. Setup

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from scipy.stats import norm

# Find the repository folder from wherever the notebook runs, and use its source code if the
# package is not installed.
ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "configs" / "base.yaml").exists())
try:
    import mfg  # noqa: F401
except ImportError:
    sys.path.insert(0, str(ROOT / "src"))

from mfg.benchmarks import lambda_n, merton_pi, population_mean, population_var
from mfg.params import Params
from mfg.pinn.nets import pick_device
from mfg.pinn.nplayer_io import RunFolder, load_solution

warnings.filterwarnings("ignore", message="Attempting to run cuBLAS")

RUN = "main"  # results/nplayer/<RUN>/, the run_name of configs/nplayer.yaml
RUN_DIR = ROOT / "results" / "nplayer" / RUN
ANALYSIS = RUN_DIR / "analysis"
assert RUN_DIR.exists(), f"nothing in {RUN_DIR}: run scripts/run_nplayer.py first"
assert ANALYSIS.exists(), f"no analysis in {ANALYSIS}: run scripts/analyze_nplayer.py first"

cfg = __import__("yaml").safe_load((RUN_DIR / "config.yaml").read_text(encoding="utf-8"))
p = Params(**cfg["params"])
summary = json.loads((ANALYSIS / "summary.json").read_text(encoding="utf-8"))
N_VALUES = [N for N in cfg["N"] if RunFolder(RUN_DIR, N, cfg["seeds"][0]).final_path.exists()]
print(f"run '{RUN}' at commit {summary['git_commit']}: lam = {p.lam}, N = {N_VALUES}")

In [ ]:
# Plot style: the network in blues (light = small N, dark = large N), references dashed orange.
INK, MUTED, BAND = "#0b0b0b", "#898781", "#f0efec"
NETWORK, REFERENCE = "#2a78d6", "#eb6834"
N_RAMP = ["#7fb0ea", "#5a97e3", "#3480dd", "#1c5cab", "#0d366b"]
plt.rcParams.update(
    {
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#c3c2b7",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "grid.linewidth": 0.6,
        "axes.titlesize": 11,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "lines.linewidth": 1.8,
        "legend.frameon": False,
    }
)


def n_colour(N):
    order = sorted(N_VALUES)
    return N_RAMP[min(order.index(N) * len(N_RAMP) // max(len(order), 1), len(N_RAMP) - 1)]


def show(name):
    # A figure saved by scripts/analyze_nplayer.py.
    display(Image(filename=str(ANALYSIS / name)))

## 2. The game, in one screen

Investor $i$ of $N$ has wealth $X^i$ and puts $\pi^i$ dollars in the stock, with their own luck:
$dX^i = (r X^i + \pi^i(\mu - r))\,dt + \sigma \pi^i\, dW^i$. Everyone sees everyone's wealth and
maximises the expected utility of their final wealth, penalised for ending below the group's
mean $\bar X_T$ (which includes themselves):

$$
U(x, \bar x) = -e^{-\gamma x} - \frac{\lambda}{2}\big((\bar x - x)^+\big)^2 .
$$

Written with the mean $\bar y$ of my $N - 1$ rivals, the penalty is the one-dimensional one with
$\lambda_N = \lambda (1 - 1/N)^2$ (`lambda_n` in `benchmarks.py`): with $N = 2$ it weighs a
quarter of $\lambda$, with $N = 10$ 81%. The network learns $v = \log(-V)$ as a function of
$(t, x, y)$, my wealth and the rivals' wealths, from the HJB equation

$$
v_t + r x v_x - \frac{\kappa^2}{2}\frac{v_x^2}{v_{xx} + v_x^2}
+ \sum_{j=1}^{N-1}\Big[(r y_j + \pi_j(\mu - r))\,v_{y_j} + \frac{\sigma^2}{2}\pi_j^2\big(v_{y_j y_j} + v_{y_j}^2\big)\Big] = 0,
\qquad v(T, x, y) = \log\!\Big(e^{-\gamma x} + \frac{\lambda_N}{2}\big((\bar y - x)^+\big)^2\Big),
$$

where $\pi_j$ is what rival $j$ invests: the same rule read from its seat. My rule is
$\pi^*(t, x, y) = -\frac{\mu - r}{\sigma^2}\,\frac{v_x}{v_{xx} + v_x^2}$. The rounds: the rivals
play Merton, the network learns the best response, the rivals switch to a frozen copy of it, and
so on until the rule only changes by training noise. There is no Fokker-Planck equation: the
group is measured by playing games (Monte Carlo).

## 3. The rounds

**What is plotted.** For each N, the change of the rule between rounds, $\Delta_k$ = mean
$|\pi_k - \pi_{k-1}|$ over 20 000 fixed check states (realistic groups, $t \le 0.9$), on a log
scale. Round 1 is the best response to Merton; the rounds stop when $\Delta$ no longer halves
(from round 3) and the result is accepted if the last $\Delta$ is below 0.02.

In [ ]:
rows = {r["N"]: r for r in summary["table"] if isinstance(r["N"], int)}
fig, ax = plt.subplots(figsize=(7, 3.6))
for N in N_VALUES:
    status = RunFolder(RUN_DIR, N, cfg["seeds"][0]).status()
    deltas = status.get("deltas", [])
    ax.semilogy(range(1, len(deltas) + 1), deltas, marker="o", color=n_colour(N), label=f"N = {N}")
ax.axhline(cfg["settings"]["accept_delta"], color=MUTED, lw=0.8)
ax.text(1.0, cfg["settings"]["accept_delta"], "accepted below this", color=MUTED, fontsize=9, va="bottom")
ax.set(title="Change of the rule between rounds", xlabel="round k", ylabel=r"$\Delta_k$ (mean $|\pi_k - \pi_{k-1}|$)")
ax.legend()
fig.tight_layout()

print(f"{'N':>3} {'rounds':>6} {'last delta':>10} {'accepted':>8} {'minutes':>8}  L-BFGS guard kept")
for N in N_VALUES:
    folder = RunFolder(RUN_DIR, N, cfg["seeds"][0])
    history, status = folder.history(), folder.status()
    kept = ", ".join(h["guard_winner"] for h in history)
    print(f"{N:>3} {status['rounds_done']:>6} {status['deltas'][-1]:>10.4f} {str(status['status'] == 'accepted'):>8} "
          f"{sum(h['seconds'] for h in history) / 60:>8.1f}  {kept}")

**What to look at.** $\Delta_1$ is large (Merton is far from the equilibrium just below the
mean); the later ones should fall fast and then flatten at the training noise, a few hundredths
at most. The 1-D loop contracted about 15 times per round, so two or three rounds should be
enough; a run that reaches 6 rounds or ends above 0.02 is reported as not accepted. The guard
column says whether L-BFGS improved the network on fresh points in each round.

## 4. The checks

**C2, the exact game.** With the original utility $-\exp(-\gamma(x - \eta \bar x))$,
$\eta = 0.5$, everyone plays Merton with risk aversion $\gamma(1 - \eta/N)$: with $N = 5$,
$\pi(0) = 1.4703 / 0.9 = 1.6337$, and the rounds must stop at once. `scripts/run_nplayer.py
--check c2` trains it and compares with the exact rule and value on the check states.

**What is plotted.** The error of $\pi$ and the relative error of $V$ at every check state,
against time, with the targets (mean 0.02 and max 0.07 for $\pi$, 0.5% for $V$).

In [ ]:
c2_path = ROOT / "results" / "nplayer" / "checks" / "c2" / f"N{cfg['checks']['c2']['N']:02d}_seed{cfg['seeds'][0]}" / "c2_errors.json"
if c2_path.exists():
    c2 = json.loads(c2_path.read_text(encoding="utf-8"))
    print(json.dumps(c2, indent=1))
    show("fig1_c2_check.png")
else:
    print("C2 not run yet: python scripts/run_nplayer.py --check c2")

**What to look at.** Both errors should stay below their targets at every time, and the check
should have stopped in round 3 or 4 (the exact rule does not change after round 1). An error that
grows towards $t = 0$ would point at the rivals' terms, the only ones that carry the equation
backward through the other wealths.

**C3, the rule at $T$.** At the final time the rule is exact and known
(`pi_terminal` with $\lambda_N$ and $\bar y$). Every round records the error of the network's
rule at $t = T$ on the wealths of the check states, outside the kink band (0.05) and further than
0.3 from the rivals' mean:

In [ ]:
print(f"{'N':>3} {'mean error, away':>17} {'max, away':>10} {'mean error, far':>16} {'max, far':>9}")
for N in N_VALUES:
    c3 = RunFolder(RUN_DIR, N, cfg["seeds"][0]).history()[-1]["c3"]
    print(f"{N:>3} {c3['mean_away']:>17.3f} {c3['max_away']:>10.3f} {c3['mean_far']:>16.3f} {c3['max_far']:>9.3f}")

As in 1-D, the network cannot follow the kink exactly: the error just outside the band can reach
tenths, and it should fall to a few hundredths further away ("far").

## 5. My rule, by N

**What is plotted.** $\pi$ against my wealth at $t = 0$, $0.5$ and $0.9$, one curve per N, with
my rivals placed at the quantiles of the population at that time
($y_j = \bar y + s_t\,\Phi^{-1}((j - \tfrac12)/(N - 1))$, a realistic group; the grey line is
their mean). Dashed: the mean-field rule from the 1-D finite-difference equilibrium at
$\lambda = 2$, the $N \to \infty$ limit.

In [ ]:
show("fig2_rule_by_n.png")

**What to look at.** Far above the rivals' mean every curve is Merton (1.47 at $t = 0$). Just
below it the rule drops: investors protect their place. The drop should be shallower the smaller
N is, for two reasons that the next figures try to separate: the penalty against my rivals is
weaker ($\lambda_N < \lambda$), and before $T$ the rivals' final mean is uncertain, which smooths
the cliff. With $N = 10$ and $20$ the curves should approach the dashed mean-field line.

The same rule, live from a saved network: how my position reacts when the rivals move. Here the
rivals' mean is shifted, with their spread kept.

In [ ]:
N = N_VALUES[-1]
sol = load_solution(RunFolder(RUN_DIR, N, cfg["seeds"][0]).final_path, pick_device())
t = 0.5
x = np.linspace(0.0, 2.2, 221)
j = np.arange(1, N)
quantiles = np.sqrt(population_var(t, p)) * norm.ppf((j - 0.5) / (N - 1))

fig, ax = plt.subplots(figsize=(7, 3.6))
for colour, shift in zip(["#9cc3f0", "#3480dd", "#0d366b"], [-0.2, 0.0, 0.2]):
    ybar = population_mean(t, p) + shift
    ax.plot(x, sol.position(t, x, ybar + quantiles), color=colour, label=f"rivals' mean {ybar:.2f}")
    ax.axvline(ybar, color=colour, lw=0.8, ls=":")
ax.axhline(merton_pi(t, p), color=MUTED, lw=0.8)
ax.text(0.0, merton_pi(t, p) + 0.03, "Merton", color=MUTED, fontsize=9)
ax.set(title=f"N = {N}, t = {t}: the rivals' mean moves (dotted lines)", xlabel="my wealth x", ylabel="position pi")
ax.legend(loc="lower right")
fig.tight_layout()

The dip follows the rivals' mean: what matters is my place in the ranking, not my wealth alone.

## 6. The ranking: a tight or a spread-out group

**What is plotted.** The largest N, $t = 0.5$: $\pi$ against my wealth with the rivals' mean
fixed and the group's width multiplied by 0.5, 1 and 1.5.

In [ ]:
show("fig3_ranking.png")

**What to look at.** A tighter group means a more certain bar to beat, so the dip should be
sharper; a spread-out group smooths it. This dependence on the spread is what the mean-field
(a single number for the crowd) cannot show. Differences of a few hundredths are within the
accuracy of the networks.

## 7. The group, by N

**What is plotted.** Against $1/N$: the final mean of the group, the spread of final wealth and
the crowding index, from Monte Carlo games played with each network (`montecarlo.py`:
100 000 investors per N, the same random numbers for every N). The crowding index is the share of
investors who end within 0.1 of their group's mean, divided by the same share without the penalty
(2 Φ(0.1 / (s_T √(1 − 1/N))) − 1: 30.3% for N = 2, 21.7% for N = ∞), so groups of different
sizes can be compared. Dashed: the mean-field at $\lambda_N$ for each N, which carries only the
"my wealth moves the mean" effect, and at $1/N = 0$ the mean-field at $\lambda$ itself.

In [ ]:
show("fig4_group_by_n.png")
keys = ["N", "lam_N", "mean", "mean_stderr", "std", "crowding_index", "rounds", "last_delta", "accepted", "minutes"]
print("  ".join(f"{k:>14}" for k in keys))
for r in summary["table"]:
    print("  ".join(f"{(f'{r[k]:.4f}' if isinstance(r.get(k), float) else str(r.get(k, ''))):>14}" for k in keys))

**What to look at.**

- **Final mean.** The penalty lowers it (people play safe below the mean); with few investors the
  penalty against the rivals is weaker, so the mean should rise towards the $\lambda = 0$ value
  1.110 as $1/N$ grows. By hand, the $\lambda_N$ curve gives about +0.027, +0.013 and +0.007 over
  the mean-field for N = 2, 3 and 5, above the Monte Carlo error (~0.001); for N = 10 and 20 the
  difference (+0.003, +0.002) is of the order of the accuracy, where the mean-field already holds.
- **Spread and crowding.** Where the network's points sit relative to the dashed curve is the
  effect of the uncertain bar: a deviation larger than the spread between seeds (the error bars,
  if a second seed was run) is a result; otherwise it stays a hypothesis.

## 8. What it costs

**What is plotted.** Left, the grid points a finite-difference solver would need per instant
with 800 points per wealth, $800^N$ (N = 3 is already 4 GB per instant, N = 4 is 3 TB). Right,
the measured training time of the network, all rounds.

In [ ]:
show("fig5_cost.png")

**What to look at.** The network's cost grows with N because the equation needs one second
derivative per rival (N − 1 extra passes per step), but it stays within hours where a grid is
impossible from N = 4. The accuracy, though, is only what the checks above certify.

## 9. Cheat sheet

| Call | What it gives |
|---|---|
| `load_solution(path)` | a saved network as an `NPlayerSolution` (`results/nplayer/<run>/N05_seed0/final.pt`) |
| `sol.value(t, x, y)` | $V(t, x, y)$; `t` and `x` broadcast like `t + x`, `y` has the N − 1 rivals on its last axis |
| `sol.position(t, x, y)` | $\pi^*(t, x, y)$, the dollars in the stock |
| `sol.residual(t, x, y)` | the HJB residual with the rivals playing the same rule: near 0 at an equilibrium |
| `simulate_games(sol.position, p, N, n_games, n_steps, seed)` | games under the rule (`mfg.montecarlo`) |
| `RunFolder(run_dir, N, seed).history()` / `.status()` | what every round recorded |
| `python scripts/run_nplayer.py --plan` | what is trained and what is missing |